## Prerequisities

In [19]:
!pip install gymnasium
!pip install pygame
!pip install stable-baselines3
!pip install torch
!pip install numpy

In [20]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from gymnasium.wrappers import RecordVideo
import pygame
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
import random

## CartPole Maze Env  
[CartPole — Gymnasium Classic Control](https://gymnasium.farama.org/environments/classic_control/cart_pole/)


In [21]:
env = gym.make("CartPole-v1", render_mode="human")
print(env.observation_space)


Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)


## Random tutorial

In [22]:
obs, info = env.reset()

for step in range(200):
    action = env.action_space.sample()  # Take a random action
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"Step: {step+1}, Action: {action}, Reward: {reward}, Obs: {obs}")
    if terminated or truncated:
        print(f"Episode ended at step {step+1}")
        obs, info = env.reset()

env.close()


Step: 1, Action: 1, Reward: 1.0, Obs: [ 0.04386358  0.15756655 -0.02127467 -0.26729658]
Step: 2, Action: 0, Reward: 1.0, Obs: [ 0.04701491 -0.03724542 -0.02662061  0.01860094]
Step: 3, Action: 0, Reward: 1.0, Obs: [ 0.04627    -0.23197569 -0.02624859  0.3027673 ]
Step: 4, Action: 1, Reward: 1.0, Obs: [ 0.04163049 -0.03648967 -0.02019324  0.0019231 ]
Step: 5, Action: 1, Reward: 1.0, Obs: [ 0.04090069  0.15891598 -0.02015478 -0.297062  ]
Step: 6, Action: 0, Reward: 1.0, Obs: [ 0.04407901 -0.03591295 -0.02609602 -0.01080309]
Step: 7, Action: 1, Reward: 1.0, Obs: [ 0.04336075  0.15957335 -0.02631208 -0.31160405]
Step: 8, Action: 1, Reward: 1.0, Obs: [ 0.04655222  0.3550601  -0.03254416 -0.6124675 ]
Step: 9, Action: 1, Reward: 1.0, Obs: [ 0.05365342  0.5506214  -0.04479351 -0.91522044]
Step: 10, Action: 0, Reward: 1.0, Obs: [ 0.06466585  0.35613292 -0.06309792 -0.63694525]
Step: 11, Action: 1, Reward: 1.0, Obs: [ 0.07178851  0.55207545 -0.07583683 -0.9488126 ]
Step: 12, Action: 0, Reward: 1

## Train a PPO model

In [ ]:
vec_env = make_vec_env(lambda: gym.make("CartPole-v1"), n_envs=1)

# Define PPO model
model = PPO(
    "MlpPolicy",
    vec_env,
    learning_rate=2.5e-4,
    n_steps=2048,         
    batch_size=64,        
    gamma=0.99,
    gae_lambda=0.95,       
    clip_range=0.2,        
    verbose=1
)

# Train
model.learn(total_timesteps=200000)

# Save
model.save("cartpole_ppo")

Using cpu device
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 23.9     |
|    ep_rew_mean     | 23.9     |
| time/              |          |
|    fps             | 1633     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 2048     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 28.9         |
|    ep_rew_mean          | 28.9         |
| time/                   |              |
|    fps                  | 1157         |
|    iterations           | 2            |
|    time_elapsed         | 3            |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0076274774 |
|    clip_fraction        | 0.0864       |
|    clip_range           | 0.2          |
|    entropy_loss         | -0.687       |
|    explained_variance   | -0.00643     

## evaluate PPO model (reward of 500 means solved)

In [24]:
from stable_baselines3.common.evaluation import evaluate_policy

env = gym.make("CartPole-v1")
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=20)
print(f"Mean reward: {mean_reward} ± {std_reward}")


Mean reward: 500.0 ± 0.0


## PPO in action

In [25]:
model = PPO.load("cartpole_ppo")
env = gym.make("CartPole-v1", render_mode="rgb_array")
video_env = RecordVideo(env, video_folder="videos_cartpole", episode_trigger=lambda e: True, fps=50)

obs, _ = video_env.reset()
terminated, truncated = False, False
step_count = 0
max_steps = 1000
while not (terminated or truncated) and step_count < max_steps:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, _ = video_env.step(action)
    step_count += 1

video_env.close()
